## 数据处理

### 1. 读取数据集与标准化
open_asm_merra(time:str)
return total_matrix:ndarray, index:list, lon, lat, time_str

从数据集中读取数据并下采样
shape(total_matrix): (channel, latitude, longitude)

### 2. 标准化并转为 torch 张量
get_init_field

(数据 - 均值) / 标准差
```
init_field = stand(index, total_matrix)
```

转为 PyTorch 张量并添加 batch 维度，返回形状为 (1, 54, latitude, longitude)
```
init_field = torch.from_numpy(init_field).to(torch.float16).unsqueeze(0).float()
```


return init_field, init_time, index, lon, lat

init_field: (1, 54, latitude, longitude):[B, C, H, W]


## 模块分析

### 1. TransUnet

```text
输入：[B,54,H,W]

    │ 统一空间尺寸
    ▼

[B,54,576,576]

    │ 时间编码
    │ - 时间尺度归一化：hourofday, dayofweek, dayofyear
    │ - 周期特征编码：cos(2πt)、sin(2πt) 并 expand 为 [B, 6, 576, 576]
    │ - 将时间编码与通道编码合并, 形状为 [B, 60, 576, 576]
    ▼

[B,60,576,576]

    │ 卷积层
    │ - 卷积层1（Sequential）：[B, 60, 576, 576] → [B, 540, 288, 288]
    │ - 卷积层2（Bottleneck）：[B, 540, 288, 288] → [B, 1280, 144, 144]
    │ - 卷积层3（Bottleneck）：[B, 1280, 144, 144] → [B, 2560, 72, 72]
    │ - 卷积层4（Bottleneck）：[B, 2560, 72, 72] → [B, 5120, 36, 36]
    ▼

[B, 5120, 36, 36]

    │ Vit 层：[B, 5120, 36, 36] → [B, 324, 768]，表示为 token
    ▼

[B, 324, 768]

    │ project_patches_back: 将 token 线性投影为可还原为空间特征的向量
    ▼

[B, 324, 20480]

    │   - reshape: [B, 324, 20480] -> [B, 18, 18, 2, 2, 5120]
        - permute(0, 3, 5, 2, 4, 1): [B, 18, 18, 2, 2, 5120] -> [B, 5120, 2, 18, 2, 18]
        - reshape:[B, 5120, 2, 18, 2, 18] -> [B, 5120, 36, 36]
    ▼

[B, 5120, 36, 36]

    │ SingleConv
    ▼

[B, 2560, 36, 36]

    │ Up_1 解码恢复空间细节，上采样。
    │ - argument_1：[B, 2560, 36, 36]
    │ - argument_2(卷积层3(Bottleneck))：[B, 2560, 72, 72]
    │ - argument_1 上采样：[B, C_up, 72, 72]
    │ - argument_1 与 argument_2 拼接：[B, 2560+C_up, 72, 72]
    ▼

    │ Up_2 解码恢复空间细节，上采样。
    │ - argument_1：[B, 2560+C_up, 72,72]
    │ - argument_2：[B, 1280, 144, 144]
    │ - argument_1 上采样：[B, C_up, 144, 144]
    │ - argument_1 与 argument_2 拼接：[B, 1280+C_up, 144, 144]
    ▼

    │ Up_3 解码恢复空间细节，上采样。
    │ - argument_1：[B, 1280+C_up, 144,144]
    │ - argument_1 上采样：[B, C_up, 288, 288]
    │ - argument_2：[B, 540, 288, 288]
    │ - argument_1 与 argument_2 拼接：[B, 540+C_up, 288, 288]
    ▼

    │ 线性层，将 540+C_up, 288, 288 映射为 54, 576, 576
    ▼

[B, 54, 576, 576]
    │ 去掉前面添加的边界再重采样到固定尺寸 576, 576
    ▼

[B, 54, 576, 576]
```


### 2. Bottleneck

1. 卷积
2. 实例归一化
3. 残差相加
4. ReLU 引入非线性


### 3. ViT

```text
输入卷积后的特征
[B, 5120, 36, 36]
    │
    │ reshape：拆分空间维度
    ▼
[B, 5120, 2, 18, 2, 18]
    │
    │ permute(0, 3, 5, 2, 4, 1)：调整维度顺序
    │ 将四个象限中相对位置相同的四个点归入同一个 token
    │ 注意：这四个点不是相邻的 2×2 小块
    ▼
[B, 18, 18, 2, 2, 5120]
    │
    │ reshape：合并 token 位置维度和特征维度
    │ token 数量：18×18 = 324
    │ 每个 token 的特征维数：2×2×5120 = 20480
    ▼
[B, 324, 20480]
    │
    │ project_patches：线性投影，将 20480 维映射为 768 维
    ▼
[B, 324, 768]
    │
    │ 在序列最前面添加 cls_token
    │ cls_token 扩展后的形状：[B, 1, 768]
    ▼
[B, 325, 768]
    │
    │ 加上可学习的位置编码 pos_emb1D
    ▼
[B, 325, 768]
    │
    │ embedding dropout：训练时随机置零部分元素，减少过拟合
    │ 不改变张量形状
    ▼
[B, 325, 768]
    │
    │ 经过 14 层 TransformerBlock，让 token 之间交换信息
    ▼
[B, 325, 768]
    │
    │ 删除序列中的第一个 token，即 cls_token
    ▼
输出空间 token 序列
[B, 324, 768]
```

### 4. TransformerEncoder

14 层 TransformerBlock 层

### 5. TransformerBlock

```text
[B, 325, 768]

    │ MultiHeadSelfAttention, 16 个多头注意力，交换 token 之间信息
    ▼

[B,325,768]

    │ 残差相加，更新后的表示 = 原来的表示 + 根据全序列信息计算出的更新
    ▼

    │ 归一化
    ▼

    │ 每个 token 内部的特征进行组合和非线性变换
    ▼

    │ 残差相加，把前馈网络的输出加回来
    ▼

    │ 归一化
    ▼

[B, 325, 768]
```


### 6. MultiHeadSelfAttention

```text
[B, 325, 768]
    │
token个数 每个token的特征维数 768 个头
    ▼

    │ 生成 Q、K、V 所需的全部特征
    ▼

[B,325, 3 * 16 * d]
    │    16 个注意力头
    │ reshape: [B,325, 3 * 16 * d] -> [B, 325, d, 3, 16]
    ▼

[B, 325, d, 3, 16]

    │ permute: [B, 325, d, 3, 16] -> [3, B, 16, 325, d]
    ▼

[3, B, 16, 325, d]

    │ token 信息交换
    │ - 得到 Q、K、V， shape 都为 [B, 16, 325, d]
    │ - Q， K 点积后 softmax，[B, 16, 325, 325]
    │ - V 加权，[B, 16, 325, d]
    │ - permute: [B, 16, 325, d] -> [B, 325, 16, d]
    │ - 拼接，[B, 325, 16 * d]
    │ - 线性层混合各头的信息，[B, 325, 768]
    ▼

[B, 325, 768]
```

## 疑问

1. 代码选用了“500 列有效区域＋两侧各 38 列扩展＝576 列”的方案；为什么采用这组具体数值
2. 为什么同一 token 取四个象限中相对位置相同的点？若取相邻的点，是否会影响模型的训练？
3. cls_token 添加的作用是？是否对模型的训练有影响？
4. ___torch_mangle_3.Conv2d 和 ___torch_mangle_4.InstanceNorm2d 的作用和差异？
5. 上采样恢复空间时为什么要使用 卷积层3（Bottleneck）保存的特征而不是其他层？
    为了能够一步步上采样恢复空间

